# Classificador por dente — "este ápice tem lesão periapical?" — ML Odonto

1. Reaproveita o localizador de ápices e os ápices de Hanói da rodada anterior (`MyDrive/ML_Odonto_DuasEtapas`).
2. Detecta os ápices nas 705 panorâmicas do DENTEX com diagnósticos (cárie, lesão periapical, dente incluso).
3. Recorta 384 × 384 px em volta de cada ápice (a 2.800 px de largura; raiz sempre para cima) e rotula:
   lesão de Hanói ou dente com lesão periapical no DENTEX = positivo; demais ápices = negativo.
4. Treina uma ResNet34 que classifica cada recorte, reforçando lesões pequenas e o DENTEX.

Só dados públicos vêm para o Colab. Tudo fica em `MyDrive/ML_Odonto_Dentes`; se a sessão cair, rode tudo de novo.
Tempo esperado (GPU T4): ~15 min de downloads, ~5 min de ápices, ~15 min de recortes, ~40 min de treino.

### 1. GPU

In [ ]:
!nvidia-smi -L
import os; print('CPUs:', os.cpu_count())

### 2. Google Drive (onde ficam modelos e resultados)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAIDA = '/content/drive/MyDrive/ML_Odonto_Dentes'
DUAS = '/content/drive/MyDrive/ML_Odonto_DuasEtapas'
!mkdir -p "{SAIDA}" /content/scripts
!ls -la "{DUAS}"

### 3. Código do projeto (gerado a partir de `scripts/`)

In [ ]:
%%writefile /content/scripts/comum.py
"""Peças compartilhadas: carregamento, normalização, atributos por pixel e métricas."""
import xml.etree.ElementTree as ET
from pathlib import Path

import cv2
import numpy as np
from scipy import ndimage as ndi
from skimage.feature import hessian_matrix, hessian_matrix_eigvals

RAIZ = Path(__file__).resolve().parent.parent
NOSSAS = RAIZ / "Treinamento" / "extraido"
HANOI = RAIZ / "Externos" / "Hanoi"
MODELOS = RAIZ / "Modelos"
RESULTADOS = RAIZ / "Resultados"

# Todas as panorâmicas são levadas a esta largura antes de tudo: as originais têm
# ~2400-2900 px e as lesões ~60-100 px, que viram ~30-50 px aqui.
LARGURA = 1400


def ler_cinza(caminho):
    # imdecode lida com acentos no caminho (cv2.imread não, no Windows)
    dados = np.fromfile(str(caminho), np.uint8)
    img = cv2.imdecode(dados, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"não consegui ler {caminho}")
    return img


def normalizar(img, largura=None):
    """Redimensiona para `largura` (padrão LARGURA) e equaliza o contraste (CLAHE) para aproximar aparelhos diferentes."""
    largura = largura or LARGURA
    escala = largura / img.shape[1]
    interp = cv2.INTER_AREA if escala <= 1 else cv2.INTER_CUBIC
    img = cv2.resize(img, (largura, round(img.shape[0] * escala)), interpolation=interp)
    img = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)
    return img, escala


def redimensionar_mascara(m, forma):
    return cv2.resize(m, (forma[1], forma[0]), interpolation=cv2.INTER_NEAREST) > 0


NOMES_ATRIBUTOS = []
VERSAO_ATRIBUTOS = 2  # muda quando atributos() muda, para invalidar amostras em cache


def deslocar(A, dy, dx):
    """Valor de A no ponto (y+dy, x+dx), repetindo a borda fora da imagem."""
    p = max(abs(dy), abs(dx))
    Ap = np.pad(A, p, mode="edge")
    h, w = A.shape
    return Ap[p + dy:p + dy + h, p + dx:p + dx + w]


def atributos(img):
    """Pilha (H, W, F) de atributos por pixel, no estilo ilastik/Weka.

    A lesão periapical é uma área mais escura que o osso ao redor, perto da ponta
    da raiz: por isso brilho em várias escalas, contraste com a vizinhança (DoG),
    bordas, forma local (Hessiana), textura e posição na imagem.
    """
    I = img.astype(np.float32) / 255.0
    h, w = I.shape
    camadas, nomes = [I], ["brilho"]
    G = {s: cv2.GaussianBlur(I, (0, 0), s) for s in (1, 2, 4, 8, 16, 32)}
    for s in (1, 2, 4, 8, 16):
        camadas.append(G[s]); nomes.append(f"gauss_{s}")
    for a, b in ((1, 4), (2, 8), (4, 16), (8, 32)):
        camadas.append(G[a] - G[b]); nomes.append(f"dog_{a}_{b}")
    for s in (1, 2, 4, 8):
        gx = cv2.Sobel(G[s], cv2.CV_32F, 1, 0)
        gy = cv2.Sobel(G[s], cv2.CV_32F, 0, 1)
        camadas.append(np.sqrt(gx * gx + gy * gy)); nomes.append(f"gradiente_{s}")
    for s in (2, 4, 8):
        H = hessian_matrix(I, sigma=s, order="rc", use_gaussian_derivatives=False)
        l1, l2 = hessian_matrix_eigvals(H)
        camadas += [l1.astype(np.float32), l2.astype(np.float32)]
        nomes += [f"hessiana_{s}_l1", f"hessiana_{s}_l2"]
    for s in (4, 8):
        m2 = cv2.GaussianBlur(I * I, (0, 0), s)
        camadas.append(np.sqrt(np.maximum(m2 - G[s] ** 2, 0))); nomes.append(f"desvio_{s}")
    # Contexto: a lesão fica na ponta de uma raiz, então o brilho alguns pixels
    # acima/abaixo/ao lado (raiz clara? osso? ar?) diz muito sobre o pixel central.
    for s in (4, 8):
        for dy, dx in ((-40, 0), (-20, 0), (20, 0), (40, 0), (0, -25), (0, 25)):
            camadas.append(deslocar(G[s], dy, dx) - G[s]); nomes.append(f"contexto_{s}_{dy}_{dx}")
    camadas.append(G[32] - cv2.GaussianBlur(I, (0, 0), 64)); nomes.append("dog_32_64")
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    camadas += [xx / w, yy / h, np.abs(xx / w - 0.5)]
    nomes += ["pos_x", "pos_y", "dist_centro_x"]
    if not NOMES_ATRIBUTOS:
        NOMES_ATRIBUTOS.extend(nomes)
    return np.stack(camadas, axis=-1)


# ---------------------------------------------------------------- conjuntos

def nossas_imagens():
    """[(id, img_normalizada, mascara_bool)] das 20 panorâmicas marcadas do PDF."""
    saida = []
    for f in sorted((NOSSAS / "imagens").glob("*.png")):
        img, _ = normalizar(ler_cinza(f))
        m = redimensionar_mascara(ler_cinza(NOSSAS / "mascaras" / f.name), img.shape)
        saida.append((f.stem, img, m))
    return saida


def caixas_hanoi(nome):
    r = ET.parse(HANOI / "anotacoes" / "Image Annots" / f"{nome}.xml").getroot()
    caixas = []
    for o in r.iter("object"):
        b = o.find("bndbox")
        caixas.append(tuple(float(b.findtext(t)) for t in ("xmin", "ymin", "xmax", "ymax")))
    return caixas


def imagem_hanoi(nome, largura=None):
    """Imagem normalizada, máscara (elipse dentro de cada caixa) e zona incerta.

    A caixa é justa em torno da lesão, então a elipse inscrita é uma boa
    aproximação. A faixa entre a elipse e a caixa ampliada é marcada como
    incerta e não entra no treino nem como lesão nem como fundo.
    """
    img, esc = normalizar(ler_cinza(HANOI / "imagens" / "Original JPG Images" / f"{nome}.jpg"), largura)
    mascara = np.zeros(img.shape, np.uint8)
    incerta = np.zeros(img.shape, np.uint8)
    for x0, y0, x1, y1 in caixas_hanoi(nome):
        cx, cy = (x0 + x1) / 2 * esc, (y0 + y1) / 2 * esc
        rx, ry = (x1 - x0) / 2 * esc, (y1 - y0) / 2 * esc
        cv2.ellipse(incerta, (round(cx), round(cy)), (round(rx * 1.5), round(ry * 1.5)), 0, 0, 360, 1, -1)
        cv2.ellipse(mascara, (round(cx), round(cy)), (round(rx * 0.85), round(ry * 0.85)), 0, 0, 360, 1, -1)
    return img, mascara > 0, (incerta > 0) & (mascara == 0)


def nomes_hanoi():
    return sorted(p.stem for p in (HANOI / "imagens" / "Original JPG Images").glob("*.jpg"))


# ---------------------------------------------------------------- amostragem

def amostrar(F, mascara, incerta, n_pos, n_neg, rng):
    """Sorteia pixels de lesão e de fundo. Metade do fundo vem da faixa dos dentes,
    onde estão as confusões difíceis (raízes, espaços entre dentes)."""
    h, w = mascara.shape
    pos = np.flatnonzero(mascara)
    neg_ok = ~mascara & ~incerta
    faixa = np.zeros_like(neg_ok)
    faixa[int(0.25 * h):int(0.85 * h), int(0.15 * w):int(0.85 * w)] = True
    neg_todos = np.flatnonzero(neg_ok)
    neg_faixa = np.flatnonzero(neg_ok & faixa)
    escolha = lambda arr, n: rng.choice(arr, min(n, len(arr)), replace=False) if len(arr) else arr
    idx_pos = escolha(pos, n_pos)
    idx_neg = np.concatenate([escolha(neg_todos, n_neg // 2), escolha(neg_faixa, n_neg - n_neg // 2)])
    Fp = F.reshape(-1, F.shape[-1])
    X = np.concatenate([Fp[idx_pos], Fp[idx_neg]])
    y = np.concatenate([np.ones(len(idx_pos), np.uint8), np.zeros(len(idx_neg), np.uint8)])
    return X, y


# ---------------------------------------------------------------- predição

def prever_probabilidade(modelo, img, lote=200_000, devolver_atributos=False):
    F = atributos(img)
    Fp = F.reshape(-1, F.shape[-1])
    p = np.concatenate([modelo.predict_proba(Fp[i:i + lote])[:, 1] for i in range(0, len(Fp), lote)])
    p = p.reshape(img.shape).astype(np.float32)
    return (p, F) if devolver_atributos else p


def segmentar(prob, limiar, area_min):
    """Probabilidade -> máscara: suaviza, corta no limiar e descarta manchas pequenas."""
    p = cv2.GaussianBlur(prob, (0, 0), 2)
    m = (p >= limiar).astype(np.uint8)
    n, rot, st, _ = cv2.connectedComponentsWithStats(m)
    manter = np.zeros(n, bool)
    manter[1:] = st[1:, cv2.CC_STAT_AREA] >= area_min
    return manter[rot]


# ---------------------------------------------------------------- métricas

FRACAO_ACERTO = 0.2


def avaliar(pred, verdade):
    """Por imagem: acertos de pixel e de lesão.

    Uma mancha prevista só é acerto se pelo menos FRACAO_ACERTO dela cair sobre
    lesão de verdade; assim uma mancha enorme que só encosta na lesão não conta.
    Uma lesão verdadeira é achada se alguma mancha correta a toca.
    """
    inter = (pred & verdade).sum()
    nv, rv = cv2.connectedComponents(verdade.astype(np.uint8))
    np_, rp = cv2.connectedComponents(pred.astype(np.uint8))
    area = np.bincount(rp.ravel(), minlength=np_)
    sobre = np.bincount(rp[verdade], minlength=np_)
    corretas = np.flatnonzero(sobre >= FRACAO_ACERTO * np.maximum(area, 1))
    corretas = corretas[corretas > 0]
    achadas = len(set(np.unique(rv[np.isin(rp, corretas)])) - {0})
    return dict(inter=int(inter), soma=int(pred.sum() + verdade.sum()),
                lesoes=nv - 1, lesoes_achadas=achadas,
                previstas=np_ - 1, previstas_corretas=len(corretas))


def resumir(linhas):
    s = {k: sum(l[k] for l in linhas) for k in linhas[0]}
    return dict(
        dice=2 * s["inter"] / max(s["soma"], 1),
        sensibilidade=s["lesoes_achadas"] / max(s["lesoes"], 1),
        precisao=s["previstas_corretas"] / max(s["previstas"], 1),
        falsos_por_imagem=(s["previstas"] - s["previstas_corretas"]) / len(linhas),
    )

In [ ]:
%%writefile /content/scripts/unet.py
"""U-Net com codificador ResNet34 (torchvision), usada no Colab para treinar e aqui para prever.

Só depende de torch e torchvision, para o mesmo arquivo rodar nos dois lugares.
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

MEDIA = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
DESVIO = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class Bloco(nn.Sequential):
    def __init__(self, entrada, saida):
        super().__init__(
            nn.Conv2d(entrada, saida, 3, padding=1, bias=False), nn.BatchNorm2d(saida), nn.ReLU(inplace=True),
            nn.Conv2d(saida, saida, 3, padding=1, bias=False), nn.BatchNorm2d(saida), nn.ReLU(inplace=True))


class UNetResNet34(nn.Module):
    def __init__(self, pre_treinada=False):
        super().__init__()
        r = torchvision.models.resnet34(weights="IMAGENET1K_V1" if pre_treinada else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)                   # 1/2, 64
        self.pool = r.maxpool
        self.c1, self.c2, self.c3, self.c4 = r.layer1, r.layer2, r.layer3, r.layer4  # 1/4 .. 1/32
        self.d4 = Bloco(512 + 256, 256)
        self.d3 = Bloco(256 + 128, 128)
        self.d2 = Bloco(128 + 64, 64)
        self.d1 = Bloco(64 + 64, 64)
        self.d0 = Bloco(64, 32)
        self.saida = nn.Conv2d(32, 1, 1)
        self.register_buffer("media", MEDIA.clone(), persistent=False)
        self.register_buffer("desvio", DESVIO.clone(), persistent=False)

    def forward(self, x):
        """x: (N, 1, H, W) em [0, 1], H e W múltiplos de 32. Devolve logits (N, 1, H, W)."""
        x = (x.repeat(1, 3, 1, 1) - self.media) / self.desvio
        s1 = self.stem(x)
        s2 = self.c1(self.pool(s1))
        s3 = self.c2(s2)
        s4 = self.c3(s3)
        s5 = self.c4(s4)
        sobe = lambda t: F.interpolate(t, scale_factor=2, mode="bilinear", align_corners=False)
        d = self.d4(torch.cat([sobe(s5), s4], 1))
        d = self.d3(torch.cat([sobe(d), s3], 1))
        d = self.d2(torch.cat([sobe(d), s2], 1))
        d = self.d1(torch.cat([sobe(d), s1], 1))
        return self.saida(self.d0(sobe(d)))


def carregar(caminho, dispositivo="cpu"):
    ck = torch.load(caminho, map_location=dispositivo, weights_only=False)
    m = UNetResNet34()
    m.load_state_dict(ck["modelo"])
    return m.to(dispositivo).eval(), ck.get("meta", {})


@torch.no_grad()
def prever(modelo, img, espelhar=True):
    """Mapa de probabilidade (H, W) para uma imagem uint8 já normalizada (comum.normalizar).

    Com espelhar=True faz a média com a previsão da imagem espelhada (a panorâmica
    é quase simétrica), o que costuma estabilizar o resultado.
    """
    h, w = img.shape
    H, W = -(-h // 32) * 32, -(-w // 32) * 32
    x = np.pad(img, ((0, H - h), (0, W - w)), mode="reflect").astype(np.float32) / 255.0
    disp = next(modelo.parameters()).device
    t = torch.from_numpy(x)[None, None].to(disp)
    p = torch.sigmoid(modelo(t))
    if espelhar:
        p = (p + torch.sigmoid(modelo(t.flip(-1))).flip(-1)) / 2
    return p[0, 0, :h, :w].float().cpu().numpy()

In [ ]:
%%writefile /content/scripts/apices.py
"""Ápices radiculares a partir dos contornos de dentes do DENTEX (conjunto de numeração).

Primeira etapa da abordagem em duas etapas: localizar as pontas das raízes para,
depois, examinar um recorte ampliado em volta de cada uma.

No DENTEX cada dente tem um polígono, o quadrante (category_id_1: 0-3 = quadrantes
FDI 1-4) e o número (category_id_2: 0-7 = dentes 1-8). O ápice é a extremidade do
polígono do lado da raiz: para cima nos quadrantes 1 e 2 (superiores) e para baixo
nos 3 e 4 (inferiores). Molares (6-8) recebem dois ápices, um em cada metade da faixa
apical, porque têm mais de uma raiz.
"""
import json
from functools import lru_cache
from pathlib import Path

import numpy as np

import comum as c

DENTEX = c.RAIZ / "Externos" / "DENTEX"
FAIXA_APICAL = 0.12  # fração da altura do dente considerada "ponta da raiz"


def _lista(v):
    return json.loads(v) if isinstance(v, str) else v


@lru_cache(maxsize=1)
def anotacoes():
    d = json.load(open(DENTEX / "train_quadrant_enumeration.json", encoding="utf-8"))
    imagens = {str(i["id"]): i for i in d["images"]}
    dentes = {}
    for a in d["annotations"]:
        pol = np.array(_lista(a["segmentation"])[0], float).reshape(-1, 2)
        dentes.setdefault(str(a["image_id"]), []).append(
            dict(poligono=pol, quadrante=int(a["category_id_1"]) + 1, dente=int(a["category_id_2"]) + 1))
    return imagens, dentes


def apices_do_dente(pol, quadrante, dente):
    """Lista de (x, y) dos ápices de um dente, em pixels da imagem original."""
    superior = quadrante in (1, 2)
    y = pol[:, 1]
    altura = y.max() - y.min()
    ext = y.min() if superior else y.max()
    faixa = pol[np.abs(y - ext) <= FAIXA_APICAL * altura]
    if dente >= 6 and len(faixa) >= 2:
        xm = faixa[:, 0].mean()
        lados = [faixa[faixa[:, 0] <= xm], faixa[faixa[:, 0] > xm]]
        pts = []
        for lado in lados:
            if len(lado):
                k = lado[:, 1].argmin() if superior else lado[:, 1].argmax()
                pts.append(tuple(lado[k]))
        if len(pts) == 2 and abs(pts[0][0] - pts[1][0]) > 0.25 * np.ptp(pol[:, 0]):
            return pts
    k = y.argmin() if superior else y.argmax()
    return [tuple(pol[k])]


def apices_da_imagem(image_id):
    """[(x, y, quadrante, dente)] de todos os ápices de uma imagem do DENTEX."""
    _, dentes = anotacoes()
    saida = []
    for t in dentes.get(str(image_id), []):
        for x, y in apices_do_dente(t["poligono"], t["quadrante"], t["dente"]):
            saida.append((float(x), float(y), t["quadrante"], t["dente"]))
    return saida


def caminho_imagem(image_id):
    imagens, _ = anotacoes()
    return DENTEX / "enumeracao" / Path(imagens[str(image_id)]["file_name"]).name

In [ ]:
%%writefile /content/scripts/duas_etapas.py
"""Abordagem em duas etapas: localizar os ápices radiculares, depois segmentar a lesão
num recorte ampliado em volta de cada ápice.

Etapa 1 (localizador): U-Net que prevê um mapa de calor de ápices na panorâmica
normalizada a 1400 px de largura (comum.LARGURA). Treinada com os contornos de dentes
do DENTEX (apices.py).

Etapa 2 (lesão): U-Net que segmenta a lesão em recortes de RECORTE x RECORTE pixels,
tirados da panorâmica normalizada a LARGURA_RECORTE (~ resolução original, o dobro da
escala de trabalho), centrados em cada ápice detectado.

Usado no Colab (treino) e localmente (avaliação/aplicação). Só depende de comum, unet,
OpenCV, NumPy e SciPy.
"""
import cv2
import numpy as np
import torch
from scipy import ndimage as ndi

import comum as c
import unet

SIGMA_APICE = 6          # px (escala 1400) do alvo gaussiano de cada ápice
LIMIAR_APICE = 0.3       # pico mínimo do mapa de calor para virar ápice
RAIO_NMS_APICE = 10      # px (escala 1400): ápices mais próximos que isso se fundem
MAX_APICES = 60          # por imagem; uma dentição completa tem ~40 ápices
LARGURA_RECORTE = 2800   # px: largura em que os recortes são tirados
RECORTE = 256            # px (escala LARGURA_RECORTE) = 128 px na escala de trabalho
MARGEM_AUMENTO = 16      # px extras guardados em volta do recorte, para deslocamento aleatório no treino


def mapa_alvo(forma, pts, sigma=SIGMA_APICE):
    """Mapa de calor alvo: máximo de gaussianas centradas nos ápices."""
    m = np.zeros(forma, np.float32)
    r = int(4 * sigma)
    for x, y in pts:
        x0, x1 = int(max(0, x - r)), int(min(forma[1], x + r + 1))
        y0, y1 = int(max(0, y - r)), int(min(forma[0], y + r + 1))
        if x0 >= x1 or y0 >= y1:
            continue
        yy, xx = np.mgrid[y0:y1, x0:x1]
        g = np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2)).astype(np.float32)
        m[y0:y1, x0:x1] = np.maximum(m[y0:y1, x0:x1], g)
    return m


def picos(p, limiar=LIMIAR_APICE, raio=RAIO_NMS_APICE, maximo=MAX_APICES):
    """Coordenadas (x, y) dos máximos locais do mapa de calor, no máximo `maximo`, os mais fortes primeiro."""
    ps = cv2.GaussianBlur(p, (0, 0), 1.5)
    m = (ps == ndi.maximum_filter(ps, size=2 * raio + 1)) & (ps >= limiar)
    ys, xs = np.nonzero(m)
    ordem = np.argsort(-ps[ys, xs])[:maximo]
    return np.column_stack([xs[ordem], ys[ordem]]).astype(np.float32)


def detectar_apices(localizador, img_trabalho):
    """Ápices (x, y) na escala de trabalho (imagem normalizada a comum.LARGURA)."""
    return picos(unet.prever(localizador, img_trabalho))


def recorte_em(img, cx, cy, tam, preencher=0):
    """Recorte tam x tam centrado em (cx, cy), completando fora da imagem com `preencher`."""
    h, w = img.shape[:2]
    x0, y0 = int(round(cx)) - tam // 2, int(round(cy)) - tam // 2
    out = np.full((tam, tam), preencher, img.dtype)
    sx0, sy0 = max(0, x0), max(0, y0)
    sx1, sy1 = min(w, x0 + tam), min(h, y0 + tam)
    if sx1 > sx0 and sy1 > sy0:
        out[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = img[sy0:sy1, sx0:sx1]
    return out, (x0, y0)


@torch.no_grad()
def prever_recortes(modelo, recortes, lote=16, espelhar=True):
    """Probabilidades de vários recortes RECORTE x RECORTE de uma vez (em lotes), como unet.prever."""
    if not recortes:
        return []
    disp = next(modelo.parameters()).device
    saida = []
    for i in range(0, len(recortes), lote):
        x = torch.from_numpy(np.stack(recortes[i:i + lote]).astype(np.float32) / 255.0)[:, None].to(disp)
        p = torch.sigmoid(modelo(x))
        if espelhar:
            p = (p + torch.sigmoid(modelo(x.flip(-1))).flip(-1)) / 2
        saida.extend(p[:, 0].float().cpu().numpy())
    return saida


def probabilidade_duas_etapas(localizador, segmentador, original):
    """Mapa de probabilidade na escala de trabalho a partir das duas etapas.

    Fora dos recortes a probabilidade é 0: a etapa 2 só examina a vizinhança dos
    ápices detectados. Recortes sobrepostos são combinados pelo máximo.
    Devolve (prob_trabalho, apices_trabalho).
    """
    img_t, esc_t = c.normalizar(original)
    apices = detectar_apices(localizador, img_t)
    img_r, esc_r = c.normalizar(original, LARGURA_RECORTE)
    fator = esc_r / esc_t
    prob_r = np.zeros(img_r.shape, np.float32)
    recs = [recorte_em(img_r, x * fator, y * fator, RECORTE) for x, y in apices]
    probs = prever_recortes(segmentador, [r for r, _ in recs])
    for p, (_, (x0, y0)) in zip(probs, recs):
        sx0, sy0 = max(0, x0), max(0, y0)
        sx1, sy1 = min(img_r.shape[1], x0 + RECORTE), min(img_r.shape[0], y0 + RECORTE)
        if sx1 > sx0 and sy1 > sy0:
            alvo = prob_r[sy0:sy1, sx0:sx1]
            np.maximum(alvo, p[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0], out=alvo)
    prob_t = cv2.resize(prob_r, (img_t.shape[1], img_t.shape[0]), interpolation=cv2.INTER_AREA)
    return prob_t, apices

In [ ]:
%%writefile /content/scripts/dentes.py
"""Classificador por dente: "este ápice tem lesão periapical?".

Usa os ápices do localizador (duas_etapas.detectar_apices) e examina cada um num recorte com mais
contexto que o da segunda etapa (RECORTE px a LARGURA_RECORTE, ou seja, RECORTE / 2 px na escala de
trabalho). Os recortes de ápices inferiores são espelhados na vertical, para que a raiz aponte sempre
para cima: assim maxila e mandíbula compartilham o que a rede aprende.

Saída de uma panorâmica: lista de (x, y, probabilidade) na escala de trabalho (comum.LARGURA).
"""
import numpy as np
import torch
import torch.nn as nn
import torchvision

import comum as c
import duas_etapas as de

LARGURA_RECORTE = 2800   # px de largura da panorâmica de onde saem os recortes (2x a escala de trabalho)
RECORTE = 384            # px a LARGURA_RECORTE = 192 px na escala de trabalho
MARGEM = 16              # px extras guardados em volta, para deslocamento aleatório no treino
R_POS = 40               # px (escala de trabalho): ápice a esta distância de uma lesão é positivo
R_NEG = 90               # px: ápice a mais que isso de qualquer lesão é negativo (entre os dois: ignorado)
RAIO_FUSAO = 25          # px: ápices previstos mais próximos que isso viram uma detecção só


class ClassificadorDente(nn.Module):
    def __init__(self, pre_treinada=False):
        super().__init__()
        self.rede = torchvision.models.resnet34(weights="IMAGENET1K_V1" if pre_treinada else None)
        self.rede.fc = nn.Linear(512, 1)
        self.register_buffer("media", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("desvio", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1), persistent=False)

    def forward(self, x):
        """x: (N, 1, H, W) em [0, 1]. Devolve logits (N,)."""
        return self.rede((x.repeat(1, 3, 1, 1) - self.media) / self.desvio)[:, 0]


def carregar(caminho, dispositivo="cpu"):
    ck = torch.load(caminho, map_location=dispositivo, weights_only=False)
    m = ClassificadorDente()
    m.load_state_dict(ck["modelo"])
    return m.to(dispositivo).eval(), ck.get("meta", {})


def recortes_apices(img_r, apices, altura_trabalho, tam=RECORTE):
    """Recortes tam x tam (escala LARGURA_RECORTE) centrados nos ápices (escala de trabalho).

    Ápices na metade inferior da imagem são espelhados na vertical (raiz para cima).
    """
    fator = LARGURA_RECORTE / c.LARGURA
    saida = []
    for x, y in apices:
        r, _ = de.recorte_em(img_r, x * fator, y * fator, tam, 0)
        saida.append(r[::-1] if y > altura_trabalho / 2 else r)
    return saida


def rotular_apices(apices, lesoes, r_pos=R_POS, r_neg=R_NEG):
    """1 = lesão, 0 = sem lesão, -1 = ignorado. `lesoes`: caixas (x0, y0, x1, y1) na escala de trabalho.

    Distância do ápice à caixa (0 se dentro dela).
    """
    rot = np.zeros(len(apices), np.int8)
    if not len(lesoes):
        return rot
    a = np.asarray(apices, np.float32).reshape(-1, 2)
    b = np.asarray(lesoes, np.float32).reshape(-1, 4)
    dx = np.maximum(0, np.maximum(b[None, :, 0] - a[:, None, 0], a[:, None, 0] - b[None, :, 2]))
    dy = np.maximum(0, np.maximum(b[None, :, 1] - a[:, None, 1], a[:, None, 1] - b[None, :, 3]))
    d = np.sqrt(dx ** 2 + dy ** 2).min(1)
    rot[d <= r_pos] = 1
    rot[(d > r_pos) & (d <= r_neg)] = -1
    return rot


@torch.no_grad()
def classificar(modelo, recortes, lote=32, espelhar=True):
    if not recortes:
        return np.zeros(0, np.float32)
    disp = next(modelo.parameters()).device
    saida = []
    for i in range(0, len(recortes), lote):
        x = torch.from_numpy(np.stack(recortes[i:i + lote]).astype(np.float32) / 255.0)[:, None].to(disp)
        p = torch.sigmoid(modelo(x))
        if espelhar:
            p = (p + torch.sigmoid(modelo(x.flip(-1)))) / 2
        saida.append(p.float().cpu().numpy())
    return np.concatenate(saida)


def detectar(localizador, classificador, original):
    """[(x, y, p)] na escala de trabalho para uma panorâmica original (tons de cinza)."""
    img_t, _ = c.normalizar(original)
    apices = de.detectar_apices(localizador, img_t)
    img_r, _ = c.normalizar(original, LARGURA_RECORTE)
    p = classificar(classificador, recortes_apices(img_r, apices, img_t.shape[0]))
    return [(float(x), float(y), float(q)) for (x, y), q in zip(apices, p)]


def fundir(deteccoes, raio=RAIO_FUSAO):
    """Junta ápices vizinhos, mantendo o de maior probabilidade (ápices de um mesmo dente)."""
    restantes = sorted(deteccoes, key=lambda d: -d[2])
    saida = []
    for x, y, p in restantes:
        if all((x - a) ** 2 + (y - b) ** 2 > raio ** 2 for a, b, _ in saida):
            saida.append((x, y, p))
    return saida

In [ ]:
%%writefile /content/scripts/treino_dentes.py
"""Treina no Colab (GPU) o classificador por dente (dentes.py).

Reaproveita o localizador de ápices e os ápices de Hanói da rodada das duas etapas (--duas-etapas).
Fases, cada uma pulada se o resultado já estiver em --saida (permite retomar):
  1. apices_dentex_doenca.json   ápices detectados nas 705 panorâmicas do DENTEX com diagnósticos
  2. recortes/                   recortes em volta dos ápices, com rótulo (lesão / sem lesão)
  3. classificador_dentes.pt     ResNet34 que classifica cada recorte

Rótulos:
  Hanói: caixa de lesão a até R_POS px do ápice = positivo; a mais de R_NEG px de todas = negativo.
  DENTEX: ápices do dente diagnosticado com lesão periapical (polígono) = positivo; os demais
          ápices da imagem, longe deles, = negativo (dentes sem esse diagnóstico).

Uso: python treino_dentes.py --saida PASTA --duas-etapas PASTA_DUAS_ETAPAS [--its 5000]
"""
import argparse
import json
import math
import os
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from joblib import Parallel, delayed
from torch.utils.data import DataLoader, Dataset

import apices as ap
import comum as c
import dentes as dn
from unet import UNetResNet34

TOTAL = dn.RECORTE + 2 * dn.MARGEM
N_NEG_HANOI = 6     # negativos guardados por imagem de Hanói
N_NEG_DENTEX = 12   # negativos guardados por imagem do DENTEX (outro aparelho: mais valiosos)
PEQUENA = 35        # px (escala de trabalho): lesão com diâmetro equivalente menor que isso é "pequena"
DENTEX_DOENCA = c.RAIZ / "Externos" / "DENTEX"
JSON_DOENCA = DENTEX_DOENCA / "train_quadrant_enumeration_disease.json"
PASTA_DOENCA = DENTEX_DOENCA / "doenca"
if not PASTA_DOENCA.exists():  # no computador local só há as 116 panorâmicas com lesão periapical
    PASTA_DOENCA = DENTEX_DOENCA / "imagens"
LESAO_PERIAPICAL = 2  # category_id_3 no DENTEX


def log(msg):
    print(msg, flush=True)


def lesoes_dentex():
    """{arquivo: (largura, [(x, y) dos ápices dos dentes com lesão periapical], todos os ápices anotados)}."""
    d = json.load(open(JSON_DOENCA, encoding="utf-8"))
    imgs = {i["id"]: i for i in d["images"]}
    saida = {i["file_name"]: (i["width"], []) for i in d["images"]}
    for a in d["annotations"]:
        if a["category_id_3"] != LESAO_PERIAPICAL:
            continue
        pol = np.array(a["segmentation"][0], float).reshape(-1, 2)
        pts = ap.apices_do_dente(pol, int(a["category_id_1"]) + 1, int(a["category_id_2"]) + 1)
        saida[imgs[a["image_id"]]["file_name"]][1].extend(pts)
    return {k: v for k, v in saida.items() if (PASTA_DOENCA / k).exists()}


# ------------------------------------------------------------------ fase 1
def fase_apices_dentex(args, disp, saida):
    arq = saida / "apices_dentex_doenca.json"
    if arq.exists():
        log("Fase 1: ápices do DENTEX já detectados")
        return json.load(open(arq))
    ck = torch.load(Path(args.duas_etapas) / "localizador_apices.pt", map_location=disp, weights_only=False)
    loc = UNetResNet34().to(disp)
    loc.load_state_dict(ck["modelo"]); loc.eval()
    import duas_etapas as de
    res, t = {}, time.time()
    nomes = sorted(lesoes_dentex())[:args.limite or None]
    for k, nm in enumerate(nomes, 1):
        img, _ = c.normalizar(c.ler_cinza(PASTA_DOENCA / nm))
        res[nm] = de.detectar_apices(loc, img).round(1).tolist()
        if k % 100 == 0:
            log(f"  [1] {k}/{len(nomes)} ({(time.time() - t) / k:.2f}s/imagem)")
    json.dump(res, open(arq, "w"))
    log(f"Fase 1: {sum(len(v) for v in res.values())} ápices em {len(res)} panorâmicas do DENTEX")
    return res


# ------------------------------------------------------------------ fase 2
def recortes_hanoi(nome, pts, semente):
    orig = c.ler_cinza(c.HANOI / "imagens" / "Original JPG Images" / f"{nome}.jpg")
    img_t, esc = c.normalizar(orig)
    img_r, _ = c.normalizar(orig, dn.LARGURA_RECORTE)
    caixas = [(x0 * esc, y0 * esc, x1 * esc, y1 * esc) for x0, y0, x1, y1 in c.caixas_hanoi(nome)]
    rot = dn.rotular_apices(pts, caixas)
    diam = []
    for x, y in pts:  # diâmetro equivalente da caixa mais próxima (para amostrar lesões pequenas)
        if caixas:
            b = min(caixas, key=lambda b: (x - (b[0] + b[2]) / 2) ** 2 + (y - (b[1] + b[3]) / 2) ** 2)
            diam.append(2 * math.sqrt((b[2] - b[0]) * (b[3] - b[1]) * math.pi / 4 / math.pi))
        else:
            diam.append(0)
    rng = np.random.default_rng(semente)
    pos = [i for i in range(len(pts)) if rot[i] == 1]
    neg = [i for i in rng.permutation(len(pts)) if rot[i] == 0][:N_NEG_HANOI]
    sel = pos + neg
    if not sel:
        return None
    rec = dn.recortes_apices(img_r, [pts[i] for i in sel], img_t.shape[0], TOTAL)
    return np.stack(rec), np.array([1] * len(pos) + [0] * len(neg), np.uint8), np.array([diam[i] for i in sel], np.float32)


def recortes_dentex(nome, pts, largura, apices_lesao, semente):
    orig = c.ler_cinza(PASTA_DOENCA / nome)
    img_t, esc = c.normalizar(orig)
    img_r, _ = c.normalizar(orig, dn.LARGURA_RECORTE)
    caixas = [(x * esc - 20, y * esc - 20, x * esc + 20, y * esc + 20) for x, y in apices_lesao]
    rot = dn.rotular_apices(pts, caixas)
    rng = np.random.default_rng(semente)
    pos = [i for i in range(len(pts)) if rot[i] == 1]
    neg = [i for i in rng.permutation(len(pts)) if rot[i] == 0][:N_NEG_DENTEX]
    sel = pos + neg
    if not sel:
        return None
    rec = dn.recortes_apices(img_r, [pts[i] for i in sel], img_t.shape[0], TOTAL)
    return np.stack(rec), np.array([1] * len(pos) + [0] * len(neg), np.uint8), np.zeros(len(sel), np.float32)


def fase_recortes(args, apices_hanoi, apices_dentex):
    pasta = Path(args.pasta_recortes)
    pasta.mkdir(parents=True, exist_ok=True)
    pronto = pasta / "pronto.json"
    if pronto.exists():
        log("Fase 2: recortes já preparados")
        return json.load(open(pronto))
    les = lesoes_dentex()
    tarefas = [("hanoi", nm, k) for k, nm in enumerate(sorted(apices_hanoi))]
    tarefas += [("dentex", nm, k) for k, nm in enumerate(sorted(apices_dentex))]
    t, partes, bloco = time.time(), [], 300

    def um(fonte, nm, k):
        if fonte == "hanoi":
            return recortes_hanoi(nm, apices_hanoi[nm], k)
        return recortes_dentex(nm, apices_dentex[nm], *les[nm], k)

    for b in range(0, len(tarefas), bloco):
        lote = tarefas[b:b + bloco]
        res = Parallel(n_jobs=max(1, args.workers))(delayed(um)(*tf) for tf in lote)
        res = [(tf, r) for tf, r in zip(lote, res) if r is not None]
        if res:
            p = f"{b:05d}"
            np.save(pasta / f"img_{p}.npy", np.concatenate([r[0] for _, r in res]))
            np.save(pasta / f"rot_{p}.npy", np.concatenate([r[1] for _, r in res]))
            np.save(pasta / f"diam_{p}.npy", np.concatenate([r[2] for _, r in res]))
            partes.append(dict(parte=p, fonte=[tf[0] for tf, r in res for _ in range(len(r[1]))],
                               nomes=[tf[1] for tf, r in res for _ in range(len(r[1]))]))
        log(f"  [2] {min(b + bloco, len(tarefas))}/{len(tarefas)} imagens ({time.time() - t:.0f}s)")
    json.dump(partes, open(pronto, "w"))
    return partes


# ------------------------------------------------------------------ fase 3
class Recortes(Dataset):
    """Sorteio balanceado: metade com lesão; entre elas, DENTEX e lesões pequenas de Hanói reforçadas."""

    GRUPOS = (("pos_dentex", .25), ("pos_pequena", .35), ("pos_hanoi", .40), ("neg_dentex", .3), ("neg_hanoi", .7))

    def __init__(self, pasta, partes, nomes_ok, n, semente=0):
        self.img, self.idx = [], {g: [] for g, _ in self.GRUPOS}
        self.todos = []
        for k, p in enumerate(partes):
            self.img.append(np.load(Path(pasta) / f"img_{p['parte']}.npy", mmap_mode="r"))
            rot = np.load(Path(pasta) / f"rot_{p['parte']}.npy")
            diam = np.load(Path(pasta) / f"diam_{p['parte']}.npy")
            for j, (fonte, nm) in enumerate(zip(p["fonte"], p["nomes"])):
                if nm not in nomes_ok:
                    continue
                if rot[j]:
                    g = "pos_dentex" if fonte == "dentex" else ("pos_pequena" if diam[j] < PEQUENA else "pos_hanoi")
                else:
                    g = "neg_dentex" if fonte == "dentex" else "neg_hanoi"
                self.idx[g].append((k, j))
                self.todos.append((k, j, int(rot[j]), g))
        self.n, self.semente = n, semente

    def resumo(self):
        return ", ".join(f"{g} {len(v)}" for g, v in self.idx.items())

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        rng = np.random.default_rng(self.semente * 1_000_003 + i)
        positivo = rng.random() < .5
        grupos = [(g, w) for g, w in self.GRUPOS if g.startswith("pos" if positivo else "neg") and self.idx[g]]
        pesos = np.array([w for _, w in grupos]); pesos /= pesos.sum()
        g = grupos[rng.choice(len(grupos), p=pesos)][0]
        k, j = self.idx[g][rng.integers(len(self.idx[g]))]
        dy, dx = rng.integers(0, 2 * dn.MARGEM + 1, size=2)
        a = self.img[k][j, dy:dy + dn.RECORTE, dx:dx + dn.RECORTE].astype(np.float32) / 255
        if rng.random() < .5:
            a = a[:, ::-1]
        a = np.clip(a ** rng.uniform(0.7, 1.4) * rng.uniform(0.85, 1.15) + rng.uniform(-0.08, 0.08), 0, 1)
        return torch.from_numpy(a.copy())[None], torch.tensor(float(g.startswith("pos")))


def auc(y, p):
    y, p = np.asarray(y), np.asarray(p)
    if y.min() == y.max():
        return float("nan")
    ordem = np.argsort(p)
    r = np.empty(len(p)); r[ordem] = np.arange(1, len(p) + 1)
    n1 = y.sum(); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def avaliar(modelo, ds, disp, lote=96):
    modelo.eval()
    m = dn.MARGEM
    ps = []
    with torch.no_grad():
        for b in range(0, len(ds.todos), lote):
            x = np.stack([ds.img[k][j, m:m + dn.RECORTE, m:m + dn.RECORTE] for k, j, _, _ in ds.todos[b:b + lote]])
            x = torch.from_numpy(x.astype(np.float32) / 255)[:, None].to(disp)
            with torch.autocast(device_type=disp, dtype=torch.float16, enabled=disp == "cuda"):
                ps.append(torch.sigmoid(modelo(x)).float().cpu().numpy())
    modelo.train()
    p = np.concatenate(ps)
    y = np.array([t[2] for t in ds.todos]); g = np.array([t[3] for t in ds.todos])
    tudo = auc(y, p)
    dentex = auc(y[np.char.endswith(g, "dentex")], p[np.char.endswith(g, "dentex")])
    sel = (g == "pos_pequena") | (g == "neg_hanoi")
    pequenas = auc(y[sel], p[sel])
    return tudo, dentex, pequenas


def fase_classificador(args, disp, saida, partes):
    arq = saida / "classificador_dentes.pt"
    if arq.exists():
        log("Fase 3: classificador já treinado")
        return
    nomes = sorted({nm for p in partes for nm in p["nomes"]})
    rng = np.random.default_rng(3)
    hanoi = [n for n in nomes if not n.endswith(".png")]
    dentex = [n for n in nomes if n.endswith(".png")]
    val = set(rng.permutation(hanoi)[:min(200, len(hanoi) // 5)]) | set(rng.permutation(dentex)[:min(60, len(dentex) // 5)])
    ds_tr = Recortes(args.pasta_recortes, partes, set(nomes) - val, args.its * args.lote)
    ds_val = Recortes(args.pasta_recortes, partes, val, 1)
    log(f"Fase 3: treino [{ds_tr.resumo()}]")
    log(f"        validação [{ds_val.resumo()}]")
    modelo = dn.ClassificadorDente(pre_treinada=not args.sem_pre_treino).to(disp)
    opt = torch.optim.AdamW(modelo.parameters(), lr=3e-4, weight_decay=1e-4)
    agenda = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda it: min(1, (it + 1) / 200) * 0.5 * (1 + math.cos(math.pi * min(it, args.its) / args.its)))
    escal = torch.amp.GradScaler(enabled=disp == "cuda")
    carga = DataLoader(ds_tr, batch_size=args.lote, num_workers=args.workers, pin_memory=disp == "cuda")
    melhor, t, acum = -1.0, time.time(), 0.0
    modelo.train()
    for it, (x, y) in enumerate(carga, 1):
        x, y = x.to(disp), y.to(disp)
        with torch.autocast(device_type=disp, dtype=torch.float16, enabled=disp == "cuda"):
            logit = modelo(x)
        l = F.binary_cross_entropy_with_logits(logit.float(), y)
        opt.zero_grad(set_to_none=True); escal.scale(l).backward(); escal.step(opt); escal.update(); agenda.step()
        acum += l.item()
        if it % 50 == 0:
            log(f"  [3] it {it}/{args.its} perda {acum / 50:.4f} ({(time.time() - t) / 50:.2f}s/it)")
            t, acum = time.time(), 0.0
        if it % args.validar_cada == 0 or it == args.its:
            a, a_d, a_p = avaliar(modelo, ds_val, disp)
            nota = (a + a_d + a_p) / 3
            if nota > melhor:
                melhor = nota
                torch.save({"modelo": modelo.state_dict(),
                            "meta": dict(iteracao=it, auc=a, auc_dentex=a_d, auc_pequenas=a_p,
                                         recorte=dn.RECORTE, largura_recorte=dn.LARGURA_RECORTE)}, arq)
            log(f"  == [3] validação it {it}: AUC {a:.3f} | DENTEX {a_d:.3f} | lesões pequenas {a_p:.3f} (melhor média {melhor:.3f})")


def main():
    a = argparse.ArgumentParser()
    a.add_argument("--saida", required=True)
    a.add_argument("--duas-etapas", required=True, help="pasta com localizador_apices.pt e apices_hanoi.json")
    a.add_argument("--pasta-recortes", default="/content/recortes_dentes")
    a.add_argument("--its", type=int, default=5000)
    a.add_argument("--lote", type=int, default=48)
    a.add_argument("--validar-cada", type=int, default=500)
    a.add_argument("--workers", type=int, default=min(4, os.cpu_count() or 2))
    a.add_argument("--limite", type=int, default=0, help="só para testes: n imagens de cada conjunto")
    a.add_argument("--sem-pre-treino", action="store_true")
    args = a.parse_args()
    disp = "cuda" if torch.cuda.is_available() else "cpu"
    saida = Path(args.saida)
    saida.mkdir(parents=True, exist_ok=True)
    log(f"Dispositivo: {disp} | CPUs: {os.cpu_count()} | workers: {args.workers}")
    apices_hanoi = json.load(open(Path(args.duas_etapas) / "apices_hanoi.json"))
    disponiveis = set(c.nomes_hanoi())
    apices_hanoi = {k: v for k, v in apices_hanoi.items() if k in disponiveis}
    apices_dentex = fase_apices_dentex(args, disp, saida)
    if args.limite:
        apices_hanoi = dict(sorted(apices_hanoi.items())[:args.limite])
        apices_dentex = dict(sorted(apices_dentex.items())[:args.limite])
    partes = fase_recortes(args, apices_hanoi, apices_dentex)
    fase_classificador(args, disp, saida, partes)
    log(f"Fim. Modelo em {saida / 'classificador_dentes.pt'}")


if __name__ == "__main__":
    main()

### 4. Baixar e extrair o conjunto de Hanói (~850 MB)

In [ ]:
%%bash
set -e
mkdir -p /content/Externos/Hanoi/imagens /content/Externos/Hanoi/anotacoes
cd /content/Externos/Hanoi
if [ ! -d "imagens/Original JPG Images" ]; then
  wget -q -O originais.rar "https://data.mendeley.com/public-files/datasets/kx52tk2ddj/files/0d016bd4-d5ac-4363-ac35-c74d87ba4022/file_downloaded"
  wget -q -O annots.rar "https://data.mendeley.com/public-files/datasets/kx52tk2ddj/files/fa6e54cc-4094-4165-9987-3a8589213696/file_downloaded"
  which unrar > /dev/null || (apt-get -qq update && apt-get -qq install -y unrar > /dev/null)
  unrar x -o+ -inul originais.rar imagens/
  unrar x -o+ -inul annots.rar anotacoes/
  rm originais.rar annots.rar
fi
echo "imagens: $(ls 'imagens/Original JPG Images' | wc -l)  anotações: $(ls 'anotacoes/Image Annots' | wc -l)"

### 5. Baixar o conjunto de diagnósticos do DENTEX (705 panorâmicas, lido de dentro do zip remoto)

In [ ]:
!pip -q install fsspec aiohttp
import os, zipfile, time, fsspec
D = '/content/Externos/DENTEX'
os.makedirs(D + '/doenca', exist_ok=True)
z = zipfile.ZipFile(fsspec.open('https://huggingface.co/datasets/ibrahimhamamci/DENTEX/resolve/main/DENTEX/training_data.zip', block_size=2**23).open())
todos = z.namelist()
js = [n for n in todos if n.endswith('train_quadrant_enumeration_disease.json')][0]
pre = js.rsplit('/', 1)[0] + '/'
open(D + '/train_quadrant_enumeration_disease.json', 'wb').write(z.read(js))
nomes = [n for n in todos if n.startswith(pre) and '/xrays/' in n and n.endswith('.png')]
t = time.time()
for k, n in enumerate(nomes, 1):
    dst = D + '/doenca/' + n.split('/')[-1]
    if not os.path.exists(dst):
        open(dst, 'wb').write(z.read(n))
    if k % 100 == 0:
        print(k, len(nomes), f'{time.time() - t:.0f}s', flush=True)
print('DENTEX diagnósticos:', len(os.listdir(D + '/doenca')), 'imagens')

### 6. Treinar o classificador (fases já prontas no Drive são puladas)

In [ ]:
!cd /content/scripts && python -u treino_dentes.py --saida "{SAIDA}" --duas-etapas "{DUAS}" --its 5000